# LankaMart: data preparation, analysis and late-delivery model
Synthetic data only. Run `python database/seed.py` first, then run the cells top to bottom and screenshot the outputs for your evidence.

In [ ]:
import sys, sqlite3
from pathlib import Path
import pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'backend'))
conn = sqlite3.connect(ROOT / 'database' / 'lankamart.db')
df = pd.read_sql_query('SELECT * FROM v_orders', conn, parse_dates=['order_date'])
df.head()

## 1. Data quality checks (screenshot this)

In [ ]:
print('Shape:', df.shape)
print('Missing values per column:'); print(df.isna().sum())
print('Duplicate order IDs:', df['order_id'].duplicated().sum())
print('Non-positive quantity:', (df['quantity'] <= 0).sum())
print('Sales mismatch (price x qty x (1-discount)):',
      ((df['unit_price']*df['quantity']*(1-df['discount_rate'])).round(2) != df['sales_amount']).sum())
print('late_delivery matches delivery_days > 5:', ((df['delivery_days'] > 5).astype(int) == df['late_delivery']).all())

## 2. Descriptive analysis with SQL

In [ ]:
pd.read_sql_query('''SELECT month, ROUND(SUM(sales_amount)) AS sales, ROUND(SUM(profit)) AS profit, COUNT(*) AS orders
FROM v_orders GROUP BY month ORDER BY month''', conn)

In [ ]:
pd.read_sql_query('''SELECT category, ROUND(SUM(sales_amount)) AS sales, ROUND(SUM(profit)) AS profit,
ROUND(AVG(returned),3) AS return_rate FROM v_orders GROUP BY category ORDER BY sales DESC''', conn)

In [ ]:
pd.read_sql_query('''SELECT region, COUNT(*) AS orders, ROUND(AVG(delivery_days),2) AS avg_days,
ROUND(AVG(late_delivery),3) AS late_rate FROM v_orders GROUP BY region ORDER BY late_rate DESC''', conn)

## 3. Charts (Figures 3 to 5 in the report)

In [ ]:
m = df.groupby(df['order_date'].dt.to_period('M'))['sales_amount'].sum()
m.index = m.index.astype(str)
m.plot(marker='o', title='Monthly sales (LKR)'); plt.ylabel('LKR'); plt.show()
df.groupby('category')['sales_amount'].sum().sort_values().plot.barh(title='Sales by category (LKR)'); plt.show()
df['delivery_days'].plot.hist(bins=range(1,12), title='Delivery days'); plt.xlabel('Days'); plt.show()

## 4. Model: Random Forest vs a simple baseline

In [ ]:
from app.ml import train, load_training_frame
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
pipe, metrics = train(conn)
metrics

In [ ]:
data = load_training_frame(conn)
X, y = data.drop(columns='late_delivery'), data['late_delivery']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
proba = pipe.predict_proba(X_te)[:, 1]
pred = (proba >= 0.5).astype(int)
print(classification_report(y_te, pred)); print(confusion_matrix(y_te, pred)); print('ROC-AUC', round(roc_auc_score(y_te, proba), 3))
base = DummyClassifier(strategy='most_frequent').fit(X_tr, y_tr)
print('Baseline accuracy (always predict on time):', round(base.score(X_te, y_te), 3))

## 5. Limitations
Synthetic data, random (not time-based) split, few features, and `returned` is only known after delivery, so a real model should not use it. Compare with the baseline above before trusting any score.